<p style="color:red; font-size:20px;">
Shopping has become a primary channel for global commerce, but its rapid
growth has also widened the opportunities for fraudulent websites that imitate
legitimate stores, deceive consumers, and disappear before accountability is possible.
These deceptive sites often use realistic branding, secure-looking URLs, and
professional interfaces, making manual detection increasingly difficult. As a result,
automated and interpretable detection systems are now essential for protecting
consumers and platforms.
</p>

<p style="font-size:18px;">
<strong>The project produces:</strong>
</p>

<ul style="font-size:18px;">
<li>A complete exploratory and analytical study of fraud-related domain signals</li>
</ul>

The dataset contains fake (fraudulent) e-shops data together with legitimate e-shops data. The dataset is balanced and contains 1140 records of 579 fake (fraudulent) and 561 real (legitimate) online shops. Each record contains the following fields:

Online shop’s URL;
Label - {legitimate, fraudulent};
Domain length - Number of symbols in the host domain name;
Top domain length - Number of symbols in the top domain name;
Presence of prefix “www” in the active URL of the online shop, values {0 - no, 1 - yes};
Number of digits in the URL;
Number of letters in the URL;
Number of dots (.) in the URL;
Number of hyphens (-) in the URL;
Presence of credit card payment, values {0 - no, 1 - yes};
Presence of money back payment, including PayPal, Alipay, Apple Pay, Google Pay, Samsung Pay, and Amazon Pay, values {0 - no, 1 - yes};
Presence of cash on delivery payment, values {0 - no, 1 - yes};
Presence of the ability to use cryptocurrencies for payments, values {0 - no, 1 - yes};
Presence of free contact emails, including Gmail, Hotmail, Outlook, Yahoo Mail, Zoho Mail, ProtonMail, iCloud Mail, GMX Mail, AOL Mail, mail.com, Yandex Mail, Mail2World, or Tutanota, values {0 – email address not found, 1 - free email address, 2 - domain email address, 3 – other email address};
Presence of logo URL, values {0 - no, 1 - yes};
SSL certificate issuer name;
SSL certificate expire date;
SSL certificate issuer organization name;
SSL certificate issuer organization ID, values {1 - Cloudflare, Inc., 2 - Let's Encrypt, 3 - Sectigo Limited, 4 - cPanel, Inc., 5 - GoDaddy.com, Inc., 6 - Amazon, 7 - DigiCert, Inc., 8 - GlobalSign nv-sa, 9 - Google Trust Services LLC, 10 - ZeroSSL, 11 - other organization};
Indication of young domain, registered 400 days ago or later, values {0 - ‘old’ domain name, 1 - ‘young’ domain name, 2 - ‘hidden’};
Domain registration date;
Presence of TrustPilot reviews, values {0 - no, 1 - yes};
TrustPilot score, values - real number from 0 to 5 or -1 if no reviews are available;
Presence of SiteJabber reviews, values {0 - no, 1 - yes};
Presence in the standard Tranco list, values {0 - no, 1 - yes};
Tranco List rank, values - integer number from 1 to 1000000 or -1 if domain is not listed in the Tranco list.

The exploratory data analysis (EDA) focused on understanding how fraudulent and
legitimate e-commerce sites differ across structural, behavioral, and reputational
dimensions. Rather than evaluating features independently, the analysis emphasized
consistent fraud patterns emerging across the dataset.

1. Domain Structure Patterns:
Fraudulent domains tend to follow identifiable naming strategies:

    
-Shorter domain length and more compact naming
-Higher use of special characters, including dots and hyphens
-Lower presence of digits, suggesting preference toward simple, brand-like
names
-Similar top-level domain lengths across both classes, indicating that TLD alone
is not a strong discriminator
    
These patterns reflect low-cost, quickly generated domains designed for short-term
operations.
    
2. SSL Certificate Behavior:
Fraudulent sites commonly exhibit SSL characteristics aligned with minimal-effort
deployment:

-Certificates issued by free certificate authorities (e.g., Let's Encrypt) dominate
both classes, so issuer name alone is not discriminatory.
-The expiration horizon (days until certificate expiry) is shorter for many
fraudulent sites, suggesting short-lived deployments.
-“Young domain” indicators show a notable skew: fraudulent domains are
disproportionately newer.
    
Together, SSL metadata provides temporal and operational cues consistent with
fraudulent behavior.

    
3.Payment & Contact Information Signals:

Fraudulent websites often list payment and policy features aimed at creating a false
sense of legitimacy:

-High presence of credit card payment and money-back guarantee claims
-Very low presence of cryptocurrency payment acceptance (but when present,
often correlated with fraud)
-Free email contact addresses and missing logos occur more frequently among
fraudulent shops
    
These features suggest superficial attempts to mimic legitimate shops while avoiding
operational commitments.
    
4.Reputation & Traffic Signals:
    
The presence of external reputation data strongly correlates with legitimacy:

-Most fraudulent sites lack TrustPilot or SiteJabber entries
-Legitimate sites occasionally have negative TrustPilot scores, but fraudulent
sites often have no score at all
-Tranco list presence is extremely sparse overall, but when present it
overwhelmingly indicates legitimate traffic
    
These signals highlight how fraudulent websites often operate outside the
conventional web ecosystem.
    
5. Summary of Key Observations:
From the EDA, several strong fraud indicators emerged:

<b>
------Newly registered domains
------Short SSL certificate lifetimes
------Absence of external reputation signals
------Low operational transparency (logos, non-free emails, reviews)
------Simplistic or suspicious domain structures
    
These insights directly informed which engineered features were developed and which baseline features were prioritized for modeling.</b>

In [1]:
import pandas as pd

In [2]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
data = pd.read_csv("/Users/poonamkumari/Downloads/e-commerce-fraud-risk-scoring/train/fraudulent_online_shops_dataset.csv")

In [4]:
data.shape

(1140, 26)

In [5]:
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,SSL certificate expire date,Issuer organization,SSL certificate issuer organization list item,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,Oct 11 03:53:36 2023 GMT,Google Trust Services LLC,9,1,2023-05-15 03:35,0,NaN,0,0,-1
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,Jun 16 23:59:59 2024 GMT,"Cloudflare, Inc.",1,1,2023-06-18 05:43,0,NaN,0,0,-1
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,Oct 24 21:53:20 2023 GMT,Let's Encrypt,2,2,Hidden,0,-1.0,0,0,-1
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,Oct 9 15:13:00 2023 GMT,Google Trust Services LLC,9,1,2022-09-20 00:00,0,-1.0,0,0,-1
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,Oct 25 08:20:27 2023 GMT,Let's Encrypt,2,1,2023-07-27 09:05,0,NaN,0,0,-1


What Does an SSL Certificate Contain?

An SSL certificate includes:

🌐 Domain name (example.com)

🏢 Organization name (for business certificates)

🔑 Public key

📅 Validity period (expiry date)

🏛️ Certificate Authority (CA) details

🔐 Digital signature

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 26 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Online shop URL                                1140 non-null   object 
 1   Label                                          1140 non-null   object 
 2   Domain length                                  1140 non-null   int64  
 3   Top domain length                              1140 non-null   int64  
 4   Presence of prefix 'www'                       1140 non-null   int64  
 5   Number  of digits                              1140 non-null   int64  
 6   Number  of letters                             1140 non-null   int64  
 7   Number  of dots (.)                            1140 non-null   int64  
 8   Number  of hyphens (-)                         1140 non-null   int64  
 9   Presence of credit card payment                1140 

In [7]:
data['Domain registration date'].head(27)

0     2023-05-15 03:35
1     2023-06-18 05:43
2               Hidden
3     2022-09-20 00:00
4     2023-07-27 09:05
5     2023-06-15 08:46
6     2015-04-16 00:22
7     2022-09-14 00:00
8     2020-11-08 20:10
9     2012-10-23 00:00
10              Hidden
11    2015-04-10 00:00
12    2002-10-20 00:00
13    2015-11-30 13:10
14    2022-10-13 08:04
15    2015-09-14 16:00
16    2023-02-09 12:50
17    2022-10-19 02:07
18    2004-10-11 16:40
19    2023-03-24 10:15
20    2022-06-19 08:45
21    2023-05-05 07:42
22    2023-06-29 06:20
23    2022-05-30 00:00
24    2023-07-12 00:00
25    2009-01-30 17:19
26    2022-08-15 22:00
Name: Domain registration date, dtype: object

In [8]:
data['Domain registration date'].dtype

dtype('O')

In [9]:
data['Domain registration date']= pd.to_datetime(data['Domain registration date'], errors='coerce')

In [10]:
data['Domain registration date'].dtype

dtype('<M8[ns]')

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 26 columns):
 #   Column                                         Non-Null Count  Dtype         
---  ------                                         --------------  -----         
 0   Online shop URL                                1140 non-null   object        
 1   Label                                          1140 non-null   object        
 2   Domain length                                  1140 non-null   int64         
 3   Top domain length                              1140 non-null   int64         
 4   Presence of prefix 'www'                       1140 non-null   int64         
 5   Number  of digits                              1140 non-null   int64         
 6   Number  of letters                             1140 non-null   int64         
 7   Number  of dots (.)                            1140 non-null   int64         
 8   Number  of hyphens (-)                         1140 non-nu

In [12]:
data.isna()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,SSL certificate expire date,Issuer organization,SSL certificate issuer organization list item,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1135,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1136,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1137,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1138,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,True,False,False,False


In [13]:
data.isnull().sum()

Online shop URL                                    0
Label                                              0
Domain length                                      0
Top domain length                                  0
Presence of prefix 'www'                           0
Number  of digits                                  0
Number  of letters                                 0
Number  of dots (.)                                0
Number  of hyphens (-)                             0
Presence of credit card payment                    0
Presence of money back payment                     0
Presence of cash on delivery payment               0
Presence of crypto currency                        0
Presence of free contact emails                    0
Presence of logo URL                               0
SSL certificate issuer                             0
SSL certificate expire date                        0
Issuer organization                                0
SSL certificate issuer organization list item 

By default axis= 0 , here sum() is doing column wise operation

In [14]:
data.isnull().sum().sort_values(ascending=False)

TrustPilot score                                 560
Domain registration date                         134
Online shop URL                                    0
Label                                              0
Presence in the standard Tranco list               0
Presence of SiteJabber reviews                     0
Presence of TrustPilot reviews                     0
Indication of young domain                         0
SSL certificate issuer organization list item      0
Issuer organization                                0
SSL certificate expire date                        0
SSL certificate issuer                             0
Presence of logo URL                               0
Presence of free contact emails                    0
Presence of crypto currency                        0
Presence of cash on delivery payment               0
Presence of money back payment                     0
Presence of credit card payment                    0
Number  of hyphens (-)                        

In [15]:
filepath= ('/Users/poonamkumari/Downloads/e-commerce-fraud-risk-scoring/train/model_metadata.json')

In [16]:
with open(filepath , 'r') as file:
    content = file.read()

In [17]:
print(content)

{
  "model_name": "RandomForest",
  "features_used": [
    "Domain length",
    "Top domain length",
    "Number  of digits",
    "Number  of letters",
    "Number  of dots (.)",
    "Number  of hyphens (-)",
    "digit_density",
    "hyphen_density",
    "dot_density",
    "letter_ratio",
    "num_payment_methods",
    "Presence of crypto currency",
    "SSL certificate issuer organization list item",
    "days_until_ssl_expiry",
    "days_since_registration",
    "is_in_tranco",
    "tranco_rank_log",
    "has_free_email",
    "has_logo",
    "young_domain",
    "trustpilot_has_reviews",
    "TrustPilot_score_clean",
    "sitejabber_has_reviews",
    "Presence of prefix 'www' "
  ],
  "metrics": {
    "LogisticRegression": {
      "accuracy": 0.9166666666666666,
      "f1": 0.9184549356223176,
      "precision": 0.9145299145299145,
      "recall": 0.9224137931034483,
      "roc_auc": 0.9806034482758621
    },
    "RandomForest": {
      "accuracy": 0.9298245614035088,
      "f1": 0.9

In [18]:
data['TrustPilot score'].unique() 

array([ nan, -1. ,  3.2,  3.7,  4.5,  4.8,  4.6,  2.9,  3.6,  3.9,  4.9,
        4.4,  3.3,  3.1,  1.7,  2.6,  4. ,  4.7,  3.8,  1.5,  1.4,  4.1,
        2.8,  2.1,  2.7,  2.4,  2.3,  4.2,  2.2,  3.4,  2.5,  1.6,  3.5,
        1.8,  1.9,  4.3,  1. ,  2. ,  5. ,  1.3])

TrustPilot score, values - real number from 0 to 5 or -1 if no reviews are available;

In [19]:
data['TrustPilot score']= data['TrustPilot score'].fillna(-1)

In [20]:
data['TrustPilot score'].head(30)

0    -1.0
1    -1.0
2    -1.0
3    -1.0
4    -1.0
5    -1.0
6     3.2
7     3.7
8    -1.0
9     4.5
10   -1.0
11   -1.0
12    4.8
13   -1.0
14   -1.0
15    4.6
16   -1.0
17    2.9
18   -1.0
19   -1.0
20   -1.0
21   -1.0
22   -1.0
23   -1.0
24   -1.0
25    3.6
26   -1.0
27   -1.0
28   -1.0
29   -1.0
Name: TrustPilot score, dtype: float64

df.loc[row_condition, column_name]

Always remember:

Row condition comes first

Column selection comes second

In [21]:
(data['TrustPilot score'] > 5).sum()

np.int64(0)

In [22]:
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,SSL certificate expire date,Issuer organization,SSL certificate issuer organization list item,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,Oct 11 03:53:36 2023 GMT,Google Trust Services LLC,9,1,2023-05-15 03:35:00,0,-1.0,0,0,-1
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,Jun 16 23:59:59 2024 GMT,"Cloudflare, Inc.",1,1,2023-06-18 05:43:00,0,-1.0,0,0,-1
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,Oct 24 21:53:20 2023 GMT,Let's Encrypt,2,2,NaT,0,-1.0,0,0,-1
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,Oct 9 15:13:00 2023 GMT,Google Trust Services LLC,9,1,2022-09-20 00:00:00,0,-1.0,0,0,-1
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,Oct 25 08:20:27 2023 GMT,Let's Encrypt,2,1,2023-07-27 09:05:00,0,-1.0,0,0,-1


In [23]:
data['TrustPilot score'].dtype

dtype('float64')

In [24]:
data.loc[data['TrustPilot score'] > 5, 'TrustPilot score'] = -1
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,SSL certificate expire date,Issuer organization,SSL certificate issuer organization list item,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,Oct 11 03:53:36 2023 GMT,Google Trust Services LLC,9,1,2023-05-15 03:35:00,0,-1.0,0,0,-1
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,Jun 16 23:59:59 2024 GMT,"Cloudflare, Inc.",1,1,2023-06-18 05:43:00,0,-1.0,0,0,-1
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,Oct 24 21:53:20 2023 GMT,Let's Encrypt,2,2,NaT,0,-1.0,0,0,-1
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,Oct 9 15:13:00 2023 GMT,Google Trust Services LLC,9,1,2022-09-20 00:00:00,0,-1.0,0,0,-1
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,Oct 25 08:20:27 2023 GMT,Let's Encrypt,2,1,2023-07-27 09:05:00,0,-1.0,0,0,-1


In [25]:
data['TrustPilot score']

0      -1.0
1      -1.0
2      -1.0
3      -1.0
4      -1.0
       ... 
1135    4.8
1136   -1.0
1137   -1.0
1138   -1.0
1139   -1.0
Name: TrustPilot score, Length: 1140, dtype: float64

In this case domain_registeration_date is missing which is highly suspicious

Domain registration date → 1006 non-null
Total rows → 1140
Missing → 134 rows

In [26]:
percentage_of_missing_values = (134/1140)* 100

In [27]:
percentage_of_missing_values

11.75438596491228

For fraud detection, you should NOT simply fill the date blindly.

Missing domain registration date itself is a signal.

Fraudulent sites often:

Hide WHOIS information

Use privacy protection

Have incomplete metadata

So missing value = potential risk indicator.

In [28]:
data['domain_data_missing'] = data['Domain registration date'].isna()

In [29]:
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,Issuer organization,SSL certificate issuer organization list item,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank,domain_data_missing
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,Google Trust Services LLC,9,1,2023-05-15 03:35:00,0,-1.0,0,0,-1,False
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,"Cloudflare, Inc.",1,1,2023-06-18 05:43:00,0,-1.0,0,0,-1,False
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,Let's Encrypt,2,2,NaT,0,-1.0,0,0,-1,True
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,Google Trust Services LLC,9,1,2022-09-20 00:00:00,0,-1.0,0,0,-1,False
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,Let's Encrypt,2,1,2023-07-27 09:05:00,0,-1.0,0,0,-1,False


New Column-- Derived Column

In [30]:
data['domain_age_days'] = (pd.Timestamp.today() - data['Domain registration date']).dt.days

In [31]:
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,SSL certificate issuer organization list item,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank,domain_data_missing,domain_age_days
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,9,1,2023-05-15 03:35:00,0,-1.0,0,0,-1,False,1023.0
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,1,1,2023-06-18 05:43:00,0,-1.0,0,0,-1,False,989.0
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,2,2,NaT,0,-1.0,0,0,-1,True,NaN
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,9,1,2022-09-20 00:00:00,0,-1.0,0,0,-1,False,1261.0
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,2,1,2023-07-27 09:05:00,0,-1.0,0,0,-1,False,950.0


In [32]:
data['domain_age_days'].dtype

dtype('float64')

In [33]:
data['domain_age_days'] = pd.to_numeric(data['domain_age_days'] , errors= 'coerce')

In [34]:
data['domain_age_days'].dtype

dtype('float64')

In [35]:
data['domain_age_days'] = data['domain_age_days'].fillna(data['domain_age_days'].median())

In [36]:
data['Domain registration date'] = data['Domain registration date'].fillna(data['Domain registration date'].median())

In [37]:
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,SSL certificate issuer organization list item,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank,domain_data_missing,domain_age_days
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,9,1,2023-05-15 03:35:00,0,-1.0,0,0,-1,False,1023.0
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,1,1,2023-06-18 05:43:00,0,-1.0,0,0,-1,False,989.0
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,2,2,2021-08-07 13:12:00,0,-1.0,0,0,-1,True,1669.0
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,9,1,2022-09-20 00:00:00,0,-1.0,0,0,-1,False,1261.0
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,2,1,2023-07-27 09:05:00,0,-1.0,0,0,-1,False,950.0


So , Till now we changed data types , handled missing values , created new column

In [38]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 28 columns):
 #   Column                                         Non-Null Count  Dtype         
---  ------                                         --------------  -----         
 0   Online shop URL                                1140 non-null   object        
 1   Label                                          1140 non-null   object        
 2   Domain length                                  1140 non-null   int64         
 3   Top domain length                              1140 non-null   int64         
 4   Presence of prefix 'www'                       1140 non-null   int64         
 5   Number  of digits                              1140 non-null   int64         
 6   Number  of letters                             1140 non-null   int64         
 7   Number  of dots (.)                            1140 non-null   int64         
 8   Number  of hyphens (-)                         1140 non-nu

To capture domain composition more meaningfully:

digit_density = digits / domain_length
hyphen_density = hyphens / domain_length
dot_density = dots / domain_length
letter_ratio = letters / domain_length
These normalized ratios generalize patterns across domains of different lengths.

In [39]:
data = data.loc[data['Domain length'] > 0]

In [40]:
data

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,SSL certificate issuer organization list item,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank,domain_data_missing,domain_age_days
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,9,1,2023-05-15 03:35:00,0,-1.0,0,0,-1,False,1023.0
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,1,1,2023-06-18 05:43:00,0,-1.0,0,0,-1,False,989.0
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,2,2,2021-08-07 13:12:00,0,-1.0,0,0,-1,True,1669.0
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,9,1,2022-09-20 00:00:00,0,-1.0,0,0,-1,False,1261.0
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,2,1,2023-07-27 09:05:00,0,-1.0,0,0,-1,False,950.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1135,https://www.tinneystoys.com,legitimate,19,3,1,0,28,2,0,1,...,7,0,2014-05-13 16:16:00,1,4.8,0,0,-1,False,4312.0
1136,https://kalistara.lt,legitimate,12,2,0,0,16,1,0,1,...,2,0,2015-01-30 00:00:00,0,-1.0,0,0,-1,False,4051.0
1137,https://www.allkindsoftools.com,fraudulent,23,3,1,0,26,2,0,1,...,9,1,2023-06-05 00:00:00,0,-1.0,0,0,-1,False,1003.0
1138,https://www.arlton.shop,fraudulent,15,4,1,0,18,2,0,1,...,9,0,2021-08-07 13:12:00,0,-1.0,0,0,-1,True,1669.0


In [41]:
data['digit_density'] = data['Number  of digits'] / data['Domain length']

In [42]:
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,Indication of young domain,Domain registration date,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank,domain_data_missing,domain_age_days,digit_density
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,1,2023-05-15 03:35:00,0,-1.0,0,0,-1,False,1023.0,0.0
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,1,2023-06-18 05:43:00,0,-1.0,0,0,-1,False,989.0,0.0
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,2,2021-08-07 13:12:00,0,-1.0,0,0,-1,True,1669.0,0.0
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,1,2022-09-20 00:00:00,0,-1.0,0,0,-1,False,1261.0,0.0
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,1,2023-07-27 09:05:00,0,-1.0,0,0,-1,False,950.0,0.0


In [43]:
data['letter_ratio'] = data['Number  of letters'] / data['Domain length']

In [44]:
data['hyphen_density'] = data['Number  of hyphens (-)'] / data['Domain length']

In [45]:
data['hyphen_density'] = data['Number  of hyphens (-)'] / data['Domain length']

In [46]:
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,Presence of TrustPilot reviews,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank,domain_data_missing,domain_age_days,digit_density,letter_ratio,hyphen_density
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,0,-1.0,0,0,-1,False,1023.0,0.0,1.125000,0.000000
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,0,-1.0,0,0,-1,False,989.0,0.0,1.111111,0.055556
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,0,-1.0,0,0,-1,True,1669.0,0.0,1.071429,0.035714
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,0,-1.0,0,0,-1,False,1261.0,0.0,1.307692,0.000000
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,0,-1.0,0,0,-1,False,950.0,0.0,1.157895,0.000000


Two important time-based variables were constructed:

days_until_ssl_expiry

Derived from certificate expiry minus reference date

Captures operational investment and longevity

days_since_registration

Derived from domain registration date

Measures domain age, a strong indicator of legitimacy
These features translate raw date fields into actionable fraud signals.

In [47]:
data['days_until_ssl_expiry'] = data['Domain registration date'] - data['Domain registration date']

In [48]:
data = data.rename( columns = {'domain_age_days': 'days_since_registration'})

In [49]:
data.columns

Index(['Online shop URL', 'Label', 'Domain length', 'Top domain length',
       'Presence of prefix 'www' ', 'Number  of digits', 'Number  of letters',
       'Number  of dots (.)', 'Number  of hyphens (-)',
       'Presence of credit card payment', 'Presence of money back payment',
       'Presence of cash on delivery payment', 'Presence of crypto currency',
       'Presence of free contact emails', 'Presence of logo URL',
       'SSL certificate issuer', 'SSL certificate expire date',
       'Issuer organization', 'SSL certificate issuer organization list item',
       'Indication of young domain ', 'Domain registration date',
       'Presence of TrustPilot reviews', 'TrustPilot score',
       'Presence of SiteJabber reviews',
       'Presence in the standard Tranco list', 'Tranco List rank',
       'domain_data_missing', 'days_since_registration', 'digit_density',
       'letter_ratio', 'hyphen_density', 'days_until_ssl_expiry'],
      dtype='object')

In [50]:
data.loc[data['Domain length']== 0]

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,TrustPilot score,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank,domain_data_missing,days_since_registration,digit_density,letter_ratio,hyphen_density,days_until_ssl_expiry


In [51]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1140 entries, 0 to 1139
Data columns (total 32 columns):
 #   Column                                         Non-Null Count  Dtype          
---  ------                                         --------------  -----          
 0   Online shop URL                                1140 non-null   object         
 1   Label                                          1140 non-null   object         
 2   Domain length                                  1140 non-null   int64          
 3   Top domain length                              1140 non-null   int64          
 4   Presence of prefix 'www'                       1140 non-null   int64          
 5   Number  of digits                              1140 non-null   int64          
 6   Number  of letters                             1140 non-null   int64          
 7   Number  of dots (.)                            1140 non-null   int64          
 8   Number  of hyphens (-)                         1140 n

'TrustPilot score'
Missing or -1 values are standardized to NaN, allowing consistent model
interpretation.

Already handled missing values , -1 to nan will convert

In [52]:
data['TrustPilot score'] = data['TrustPilot score'].replace(-1, np.nan)

NaN= Not a number

In [53]:
data['TrustPilot score'].head(10)

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    3.2
7    3.7
8    NaN
9    4.5
Name: TrustPilot score, dtype: float64

In [54]:
data['Tranco List rank'].unique()

array([    -1,  90036, 802243, 804620,  35556, 459343, 825217, 802653,
       682488, 538286, 100803, 875089, 797093, 796877, 918108, 346606,
       750712, 964101, 481570])

Tranco List rank, values - integer number from 1 to 1000000 or -1 if domain is not listed in the Tranco list.

In [55]:
def Binary_indicator(x):
    if (x >=1) and (x <=1000000):
        return "1"
    else:
        return "0"
        
        
        

In [56]:
data['is_in_tranco']= data['Tranco List rank'].apply(Binary_indicator)

In [57]:
data.head()

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,Presence of SiteJabber reviews,Presence in the standard Tranco list,Tranco List rank,domain_data_missing,days_since_registration,digit_density,letter_ratio,hyphen_density,days_until_ssl_expiry,is_in_tranco
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,0,0,-1,False,1023.0,0.0,1.125000,0.000000,0 days,0
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,0,0,-1,False,989.0,0.0,1.111111,0.055556,0 days,0
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,0,0,-1,True,1669.0,0.0,1.071429,0.035714,0 days,0
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,0,0,-1,False,1261.0,0.0,1.307692,0.000000,0 days,0
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,0,0,-1,False,950.0,0.0,1.157895,0.000000,0 days,0


tranco_rank_log
Log-transformed rank for smoother scaling of large numeric values; defaults to
0 for missing ranks.

💡 Why Log Transformation?

Log transformation:

Compresses large numbers

Makes scale smoother

Reduces extreme differences

In [58]:
data['Tranco List rank'].dtype

dtype('int64')

In [59]:
data['Tranco List rank'] = pd.to_numeric(data['Tranco List rank'], errors= 'coerce')

In [60]:



data['tranco_rank_log'] = np.log1p(data['Tranco List rank'])

data['tranco_rank_log'] = data['tranco_rank_log'].fillna(0)

np.log1p(x) = log(1 + x)

In [63]:
data.head(5)

,Online shop URL,Label,Domain length,Top domain length,Presence of prefix 'www',Number of digits,Number of letters,Number of dots (.),Number of hyphens (-),Presence of credit card payment,...,Presence in the standard Tranco list,Tranco List rank,domain_data_missing,days_since_registration,digit_density,letter_ratio,hyphen_density,days_until_ssl_expiry,is_in_tranco,tranco_rank_log
0,https://www.allaccessorybest.com,fraudulent,24,3,1,0,27,2,0,1,...,0,-1,False,1023.0,0.0,1.125000,0.000000,0 days,0,-inf
1,https://www.b-watches.shop,fraudulent,18,4,1,0,20,2,1,0,...,0,-1,False,989.0,0.0,1.111111,0.055556,0 days,0,-inf
2,https://www.waeschenamen-windrath.de,legitimate,28,2,1,0,30,2,1,1,...,0,-1,True,1669.0,0.0,1.071429,0.035714,0 days,0,-inf
3,https://vendoprint.se,legitimate,13,2,0,0,17,1,0,1,...,0,-1,False,1261.0,0.0,1.307692,0.000000,0 days,0,-inf
4,https://www.newbikeland.com,fraudulent,19,3,1,0,22,2,0,1,...,0,-1,False,950.0,0.0,1.157895,0.000000,0 days,0,-inf


In [64]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1140 entries, 0 to 1139
Data columns (total 34 columns):
 #   Column                                         Non-Null Count  Dtype          
---  ------                                         --------------  -----          
 0   Online shop URL                                1140 non-null   object         
 1   Label                                          1140 non-null   object         
 2   Domain length                                  1140 non-null   int64          
 3   Top domain length                              1140 non-null   int64          
 4   Presence of prefix 'www'                       1140 non-null   int64          
 5   Number  of digits                              1140 non-null   int64          
 6   Number  of letters                             1140 non-null   int64          
 7   Number  of dots (.)                            1140 non-null   int64          
 8   Number  of hyphens (-)                         1140 n

In [66]:
data['TrustPilot score'].isna().sum()

np.int64(885)

In [67]:
data['Presence of free contact emails'].unique()

array([0, 1, 2, 3])

In [68]:
def has_free_email(y):
    if y==1:
        return "1"
    else:
        return "0"

In [69]:
data['has_free_email'] = data['Presence of free contact emails'].apply(has_free_email)

In [70]:
data['Presence of money back payment'].unique()

array([1, 0])